### 20 epochs

In [3]:
# ──── 1. 라이브러리 import ────────────────────────────────────────
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm import trange
import matplotlib.pyplot as plt

# ──── 2. VGG 블록 & 모델 정의 (기존 코드 동일) ────────────────────
def conv_2_block(in_dim, out_dim):
    return nn.Sequential(
        nn.Conv2d(in_dim, out_dim, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_dim, out_dim, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2, 2)
    )

def conv_3_block(in_dim, out_dim):
    return nn.Sequential(
        nn.Conv2d(in_dim, out_dim, 3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_dim, out_dim, 3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_dim, out_dim, 3, padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2, 2)
    )

class VGG16_CIFAR(nn.Module):
    def __init__(self, base_dim=64, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            conv_2_block(3, base_dim),               
            conv_2_block(base_dim, base_dim*2),      
            conv_3_block(base_dim*2, base_dim*4),    
            conv_3_block(base_dim*4, base_dim*8),    
            conv_3_block(base_dim*8, base_dim*8),    
        )
        self.classifier = nn.Sequential(
            nn.Linear(base_dim*8*1*1, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(4096, 1000),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(1000, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

# ──── 3. 설정 ───────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size   = 100
learning_rate= 2e-4
num_epochs   = 20

# ──── 4. 데이터 준비 ────────────────────────────────────────────
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),
                         (0.2470,0.2435,0.2616))
])
train_ds = torchvision.datasets.CIFAR10(root="./data", train=True,  download=True, transform=transform)
test_ds  = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=4)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=4)

# ──── 5. 모델·손실·옵티마이저·TensorBoard 준비 ─────────────────
model     = VGG16_CIFAR(base_dim=64, num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
writer    = SummaryWriter(log_dir="runs/cifar10_vgg")

# ──── 6. 학습 루프 ─────────────────────────────────────────────
global_step = 0
for epoch in range(1, num_epochs+1):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        global_step += 1

        # 배치별 loss 기록
        if global_step % 50 == 0:
            writer.add_scalar("Train/Batch_Loss", loss.item(), global_step)

    # 에폭별 평균 loss
    epoch_loss = running_loss / len(train_loader)
    writer.add_scalar("Train/Epoch_Loss", epoch_loss, epoch)

    # ──── 7. 검증(테스트) 루프 ─────────────────────────────────
    model.eval()
    correct = 0
    total   = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = outputs.max(1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

    val_acc = correct / total
    writer.add_scalar("Test/Accuracy", val_acc, epoch)

    print(f"[Epoch {epoch:02d}/{num_epochs}] "
          f"Train Loss: {epoch_loss:.4f}  Test Acc: {val_acc:.4f}")

# ──── 8. 마무리 ────────────────────────────────────────────────
writer.close()


Files already downloaded and verified
Files already downloaded and verified
[Epoch 01/20] Train Loss: 1.9604  Test Acc: 0.2700
[Epoch 02/20] Train Loss: 1.6475  Test Acc: 0.4231
[Epoch 03/20] Train Loss: 1.3234  Test Acc: 0.5666
[Epoch 04/20] Train Loss: 1.0636  Test Acc: 0.6439
[Epoch 05/20] Train Loss: 0.8668  Test Acc: 0.7174
[Epoch 06/20] Train Loss: 0.7246  Test Acc: 0.7351
[Epoch 07/20] Train Loss: 0.6191  Test Acc: 0.7552
[Epoch 08/20] Train Loss: 0.5116  Test Acc: 0.7672
[Epoch 09/20] Train Loss: 0.4462  Test Acc: 0.7803
[Epoch 10/20] Train Loss: 0.3563  Test Acc: 0.7845
[Epoch 11/20] Train Loss: 0.3006  Test Acc: 0.7783
[Epoch 12/20] Train Loss: 0.2502  Test Acc: 0.7946
[Epoch 13/20] Train Loss: 0.2179  Test Acc: 0.7914
[Epoch 14/20] Train Loss: 0.1836  Test Acc: 0.7920
[Epoch 15/20] Train Loss: 0.1610  Test Acc: 0.7989
[Epoch 16/20] Train Loss: 0.1426  Test Acc: 0.7895
[Epoch 17/20] Train Loss: 0.1172  Test Acc: 0.7921
[Epoch 18/20] Train Loss: 0.1132  Test Acc: 0.7919
[Epoch

![](https://velog.velcdn.com/images/changh2_00/post/e815a8ed-3d20-44d1-b1d0-a0a4b5df4e8d/image.png)